# TA-DA Analysis
## 1 Load model level data

In [3]:
from scipy import stats
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import re
import pandasql as ps

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Specify the experiment name and date to analyze
# Set experiment_name to None to analyze all experiments
experiment_name = "goal_vs_consent_based_analysis"
experiment_date = "20251107"  # Format: YYYYMMDD
# Note: The actual data is in the directory: consent_first_vs_goal_first_full_analysis_20251013

In [4]:
def extract_experiment_info(config_filename):
    """Extract experiment name and configuration from config filename.
    
    Example: consent_or_goal_sensitivity_analysis_(seed_2)_seed_2:_0-1000-0_20251013_165148_config.json
    Returns: ('consent_or_goal_sensitivity_analysis', '0-1000-0', '2')
    """
    # Remove _config.json suffix
    name = config_filename.replace('_config.json', '')
    
    # Pattern: {experiment_name}_(seed_{N})_seed_{N}:_{agent_config}_{timestamp}
    # Match the experiment name (everything before _(seed_)
    match = re.match(r'(.+?)_\(seed_(\d+)\)_seed_\2:_(.+?)_(\d{8}_\d{6})$', name)
    
    if match:
        exp_name = match.group(1)
        seed = match.group(2)
        agent_config = match.group(3)
        timestamp_date = match.group(4).split('_')[0]
        return exp_name, agent_config, seed, timestamp_date
    
    return None, None, None

def create_figures_directory(experiment_name, experiment_date):
    """Create figures directory for the experiment if it doesn't exist."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find the experiment directory (it might have a different name than expected)
    experiment_dir = None
    for subdir in results_dir.iterdir():
        if subdir.is_dir():
            # Check if this directory contains files matching our experiment name and date
            configs_dir = subdir / "configs"
            if configs_dir.exists():
                for config_file in configs_dir.glob("*.json"):
                    exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                    if exp_name == experiment_name and file_date == experiment_date:
                        experiment_dir = subdir
                        break
                if experiment_dir:
                    break
    
    if experiment_dir:
        figures_dir = experiment_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir
    else:
        # Fallback: create in main results directory
        figures_dir = results_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir

def load_simulation_data(experiment_name=None, experiment_date=None):
    """Load all simulation data and extract agent ratios."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    simulation_data = []
    timestamp_date = None
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config
        with open(config_file, 'r') as f:
            config = json.load(f)
        
        # Extract agent counts
        params = config['parameters']
        consent_first = params.get('ConsentFirstAgent_COUNT', 0)
        goal_first = params.get('GoalFirstAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = consent_first + goal_first + fifty_fifty
        
        # Calculate ratios
        consent_ratio = consent_first / total_agents if total_agents > 0 else 0
        goal_ratio = goal_first / total_agents if total_agents > 0 else 0
        fifty_fifty_ratio = fifty_fifty / total_agents if total_agents > 0 else 0
        
        # Find corresponding model data file
        config_name = config_file.stem
        prefix = config_name.rsplit('_', 1)[0]
        
        # Look for data files in multiple locations
        model_file = None
        agent_file = None
        
        # List of directories to check for data files
        data_dirs_to_check = []
        
        # Check main data directory first
        main_data_dir = results_dir / "data"
        if main_data_dir.exists():
            data_dirs_to_check.append(main_data_dir)
        
        # Check experiment-specific subdirectory
        if experiment_name and experiment_date:
            exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
            if exp_subdir.exists():
                exp_data_dir = exp_subdir / "data"
                if exp_data_dir.exists():
                    data_dirs_to_check.append(exp_data_dir)
        
        # Also search all subdirectories for data files that match the experiment name and date
        if experiment_name and experiment_date:
            for subdir in results_dir.iterdir():
                if subdir.is_dir():
                    exp_data_dir = subdir / "data"
                    if exp_data_dir.exists() and exp_data_dir not in data_dirs_to_check:
                        # Check if any files in this directory match our criteria
                        # We'll check by looking for files with the same prefix as our config file
                        data_dirs_to_check.append(exp_data_dir)
        
        # Find the first directory that contains the required files
        for data_dir in data_dirs_to_check:
            model_file = data_dir / f"{prefix}_model.csv"
            agent_file = data_dir / f"{prefix}_agents.csv"
            if model_file.exists() and agent_file.exists():
                break
        
        if model_file and model_file.exists():
            # Load model data
            model_df = pd.read_csv(model_file)
            agent_df = pd.read_csv(agent_file)

            # Calculate CI state ratios
            # Handle division by zero
            model_df["Consent Violation Ratio"] = model_df["Total Violated Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Fulfillment Ratio"] = model_df["Total Fulfilled Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Unrealized Ratio"] = model_df["Total Unrealized Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Deferred Ratio"] = model_df["Total Deferred Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Resource Conflict Counter Goal Accomplishment Ratio"] = model_df["Total Resource Conflicts"] / model_df["Total Resource Conflict Accomplished Counter Goals"].replace(0, np.nan)
            
            # Exclude the last early_stop_steps - 1 steps before getting final values
            # But here we should also check if no additional goals were really accomplished after the early stop steps.
            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]
                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    model_df = model_df.iloc[:-steps_to_exclude]
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
                    
            
            # Get final values (last distinct_agent_count rows after exclusion)
            final_agent_values = agent_df.iloc[-distinct_agent_count:]
            
            # Calculate agent-level metrics for each agent type
            consent_first_mask = final_agent_values['Agent Persona'] == 'ConsentFirstAgent'
            goal_first_mask = final_agent_values['Agent Persona'] == 'GoalFirstAgent'
            
            # Calculate consent-related metrics from the available columns
            # Note: The CSV has different column names than expected
            avg_accomplished_goals_consent_first_agent = final_agent_values[consent_first_mask]['Accomplished Goals'].mean() if consent_first_mask.any() else 0
            avg_accomplished_goals_goal_first_agent = final_agent_values[goal_first_mask]['Accomplished Goals'].mean() if goal_first_mask.any() else 0
            avg_remaining_goals_consent_first_agent = final_agent_values[consent_first_mask]['Remaining Goals'].mean() if consent_first_mask.any() else 0
            avg_remaining_goals_goal_first_agent = final_agent_values[goal_first_mask]['Remaining Goals'].mean() if goal_first_mask.any() else 0
            avg_resource_conflicts_consent_first_agent = final_agent_values[consent_first_mask]['Resource Conflicts'].mean() if consent_first_mask.any() else 0
            avg_resource_conflicts_goal_first_agent = final_agent_values[goal_first_mask]['Resource Conflicts'].mean() if goal_first_mask.any() else 0
            avg_counter_goal_accomplishments_consent_first_agent = final_agent_values[consent_first_mask]['Counter Conflict Goal Accomplishments'].mean() if consent_first_mask.any() else 0
            avg_counter_goal_accomplishments_goal_first_agent = final_agent_values[goal_first_mask]['Counter Conflict Goal Accomplishments'].mean() if goal_first_mask.any() else 0
            
            # Calculate consent metrics from available columns
            # Separately for R (Receiver) and G (Giver) and agent type.
            total_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R'].mean() if consent_first_mask.any() else 0
            total_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G'].mean() if consent_first_mask.any() else 0
            total_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R'].mean() if goal_first_mask.any() else 0
            total_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G'].mean() if goal_first_mask.any() else 0
            violated_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R Violated'].mean() if consent_first_mask.any() else 0
            violated_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G Violated'].mean() if consent_first_mask.any() else 0
            violated_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R Violated'].mean() if goal_first_mask.any() else 0
            violated_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G Violated'].mean() if goal_first_mask.any() else 0
            
            fulfilled_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R Fulfilled'].mean() if consent_first_mask.any() else 0
            fulfilled_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G Fulfilled'].mean() if consent_first_mask.any() else 0
            fulfilled_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R Fulfilled'].mean() if goal_first_mask.any() else 0
            fulfilled_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G Fulfilled'].mean() if goal_first_mask.any() else 0
            
            # Calculate ratios (avoid division by zero)
            avg_consent_violation_ratio_consent_first_r = (violated_consents_consent_first_r / total_consents_consent_first_r) if total_consents_consent_first_r > 0 else 0
            avg_consent_violation_ratio_consent_first_g = (violated_consents_consent_first_g / total_consents_consent_first_g) if total_consents_consent_first_g > 0 else 0
            avg_consent_violation_ratio_goal_first_r = (violated_consents_goal_first_r / total_consents_goal_first_r) if total_consents_goal_first_r > 0 else 0
            avg_consent_violation_ratio_goal_first_g = (violated_consents_goal_first_g / total_consents_goal_first_g) if total_consents_goal_first_g > 0 else 0
            
            avg_consent_fulfillment_ratio_consent_first_r = (fulfilled_consents_consent_first_r / total_consents_consent_first_r) if total_consents_consent_first_r > 0 else 0
            avg_consent_fulfillment_ratio_consent_first_g = (fulfilled_consents_consent_first_g / total_consents_consent_first_g) if total_consents_consent_first_g > 0 else 0
            avg_consent_fulfillment_ratio_goal_first_r = (fulfilled_consents_goal_first_r / total_consents_goal_first_r) if total_consents_goal_first_r > 0 else 0
            avg_consent_fulfillment_ratio_goal_first_g = (fulfilled_consents_goal_first_g / total_consents_goal_first_g) if total_consents_goal_first_g > 0 else 0
        
            
            # Resource conflict counter goal accomplishment ratio
            avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent = (avg_resource_conflicts_consent_first_agent / avg_counter_goal_accomplishments_consent_first_agent) if avg_counter_goal_accomplishments_consent_first_agent > 0 else 0
            avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent = (avg_resource_conflicts_goal_first_agent / avg_counter_goal_accomplishments_goal_first_agent) if avg_counter_goal_accomplishments_goal_first_agent > 0 else 0
            
            # Calculate interaction and timing metrics
            avg_finished_step_consent_first_agent = final_agent_values[consent_first_mask]['Finished Step'].mean() if consent_first_mask.any() else 0
            avg_finished_step_goal_first_agent = final_agent_values[goal_first_mask]['Finished Step'].mean() if goal_first_mask.any() else 0
            avg_longest_idle_time_consent_first_agent = final_agent_values[consent_first_mask]['Longest Idle Time'].mean() if consent_first_mask.any() else 0
            avg_longest_idle_time_goal_first_agent = final_agent_values[goal_first_mask]['Longest Idle Time'].mean() if goal_first_mask.any() else 0
            avg_distinct_agents_interacted_r_consent_first_agent = final_agent_values[consent_first_mask]['Number of Distinct Agents Interacted as R'].mean() if consent_first_mask.any() else 0
            avg_distinct_agents_interacted_r_goal_first_agent = final_agent_values[goal_first_mask]['Number of Distinct Agents Interacted as R'].mean() if goal_first_mask.any() else 0
            avg_distinct_agents_interacted_g_consent_first_agent = final_agent_values[consent_first_mask]['Number of Distinct Agents Interacted as G'].mean() if consent_first_mask.any() else 0
            avg_distinct_agents_interacted_g_goal_first_agent = final_agent_values[goal_first_mask]['Number of Distinct Agents Interacted as G'].mean() if goal_first_mask.any() else 0
            # New: total idle time per agent
            avg_total_idle_time_consent_first_agent = final_agent_values[consent_first_mask]['Total Idle Time'].mean() if consent_first_mask.any() else 0
            avg_total_idle_time_goal_first_agent = final_agent_values[goal_first_mask]['Total Idle Time'].mean() if goal_first_mask.any() else 0
            
            # Calculate steps for this run as the last value of the Step/index column
            if not model_df.empty:
                if 'Step' in model_df.columns:
                    avg_steps_overall = int(pd.to_numeric(model_df['Step'], errors='coerce').dropna().iloc[-1])
                else:
                    first_col = model_df.columns[0]
                    avg_steps_overall = int(pd.to_numeric(model_df[first_col], errors='coerce').dropna().iloc[-1])
            else:
                avg_steps_overall = np.nan
            final_values = model_df.iloc[-1]
            
            simulation_data.append({
                'experiment_name': exp_name,
                'agent_config': agent_config,
                'seed': seed,
                'config_name': config_name,
                'consent_first_count': consent_first,
                'goal_first_count': goal_first,
                'fifty_fifty_count': fifty_fifty,
                'total_agents': total_agents,
                'accomplished_goals': final_values['Total Accomplished Goals'],
                'remaining_goals': final_values['Total Remaining Goals'],
                'violated_consents': final_values['Total Violated Consents'],
                'total_consents': final_values['Total Consent Activations'],
                'resource_conflicts': final_values['Total Resource Conflicts'],
                'counter_goal_accomplishments': final_values['Total Resource Conflict Accomplished Counter Goals'],
                'consent_violation_ratio': final_values['Consent Violation Ratio'],
                'consent_fulfillment_ratio': final_values['Consent Fulfillment Ratio'],
                'consent_unrealized_ratio': final_values['Consent Unrealized Ratio'],
                'consent_deferred_ratio': final_values['Consent Deferred Ratio'],
                'resource_conflict_counter_goal_accomplishment_ratio': final_values['Resource Conflict Counter Goal Accomplishment Ratio'],
                'max_steps': config.get('max_steps', 1000),
                'avg_steps_overall': avg_steps_overall,
                'avg_accomplished_goals_consent_first_agent': avg_accomplished_goals_consent_first_agent,
                'avg_accomplished_goals_goal_first_agent': avg_accomplished_goals_goal_first_agent,
                'avg_remaining_goals_consent_first_agent': avg_remaining_goals_consent_first_agent,
                'avg_remaining_goals_goal_first_agent': avg_remaining_goals_goal_first_agent,
                # R (Receiver) specific metrics
                'avg_total_consents_consent_first_r': total_consents_consent_first_r,
                'avg_total_consents_goal_first_r': total_consents_goal_first_r,
                'avg_violated_consents_consent_first_r': violated_consents_consent_first_r,
                'avg_violated_consents_goal_first_r': violated_consents_goal_first_r,
                'avg_fulfilled_consents_consent_first_r': fulfilled_consents_consent_first_r,
                'avg_fulfilled_consents_goal_first_r': fulfilled_consents_goal_first_r,
                'avg_consent_violation_ratio_consent_first_r': avg_consent_violation_ratio_consent_first_r,
                'avg_consent_violation_ratio_goal_first_r': avg_consent_violation_ratio_goal_first_r,
                'avg_consent_fulfillment_ratio_consent_first_r': avg_consent_fulfillment_ratio_consent_first_r,
                'avg_consent_fulfillment_ratio_goal_first_r': avg_consent_fulfillment_ratio_goal_first_r,
                
                # G (Giver) specific metrics
                'avg_total_consents_consent_first_g': total_consents_consent_first_g,
                'avg_total_consents_goal_first_g': total_consents_goal_first_g,
                'avg_violated_consents_consent_first_g': violated_consents_consent_first_g,
                'avg_violated_consents_goal_first_g': violated_consents_goal_first_g,
                'avg_fulfilled_consents_consent_first_g': fulfilled_consents_consent_first_g,
                'avg_fulfilled_consents_goal_first_g': fulfilled_consents_goal_first_g,
                'avg_consent_violation_ratio_consent_first_g': avg_consent_violation_ratio_consent_first_g,
                'avg_consent_violation_ratio_goal_first_g': avg_consent_violation_ratio_goal_first_g,
                'avg_consent_fulfillment_ratio_consent_first_g': avg_consent_fulfillment_ratio_consent_first_g,
                'avg_consent_fulfillment_ratio_goal_first_g': avg_consent_fulfillment_ratio_goal_first_g,
                
                # General agent metrics
                'avg_resource_conflicts_consent_first_agent': avg_resource_conflicts_consent_first_agent,
                'avg_resource_conflicts_goal_first_agent': avg_resource_conflicts_goal_first_agent,
                'avg_counter_goal_accomplishments_consent_first_agent': avg_counter_goal_accomplishments_consent_first_agent,
                'avg_counter_goal_accomplishments_goal_first_agent': avg_counter_goal_accomplishments_goal_first_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent,
                
                # Interaction and timing metrics
                'avg_finished_step_consent_first_agent': avg_finished_step_consent_first_agent,
                'avg_finished_step_goal_first_agent': avg_finished_step_goal_first_agent,
                'avg_longest_idle_time_consent_first_agent': avg_longest_idle_time_consent_first_agent,
                'avg_longest_idle_time_goal_first_agent': avg_longest_idle_time_goal_first_agent,
                'avg_distinct_agents_interacted_r_consent_first_agent': avg_distinct_agents_interacted_r_consent_first_agent,
                'avg_distinct_agents_interacted_r_goal_first_agent': avg_distinct_agents_interacted_r_goal_first_agent,
                'avg_distinct_agents_interacted_g_consent_first_agent': avg_distinct_agents_interacted_g_consent_first_agent,
                'avg_distinct_agents_interacted_g_goal_first_agent': avg_distinct_agents_interacted_g_goal_first_agent,
                # New: total idle time
                'avg_total_idle_time_consent_first_agent': avg_total_idle_time_consent_first_agent,
                'avg_total_idle_time_goal_first_agent': avg_total_idle_time_goal_first_agent,
            })
        else:
            print(f"Warning: Model data file not found for {config_name}")
    
    return pd.DataFrame(simulation_data), timestamp_date

def create_agent_ratio_analysis(experiment_name=None, experiment_date=None):
    """Create comprehensive analysis of how metrics change with agent ratios.
    
    This function averages results across all seeds for each experiment configuration.
    """
    print(f"Analyzing experiment: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    
    # Load data
    df, timestamp_date = load_simulation_data(experiment_name=experiment_name, experiment_date=experiment_date)
    
    if df.empty:
        print("No simulation data found!")
        return
    
    # Group by experiment_name and agent_config, then calculate mean and std
    metrics_to_average = [
        'consent_first_count', 'goal_first_count', 'fifty_fifty_count', 'total_agents',
        'accomplished_goals', 'remaining_goals', 'violated_consents', 'total_consents',
        'resource_conflicts', 'counter_goal_accomplishments',
        'consent_violation_ratio', 'consent_fulfillment_ratio', 
        'consent_unrealized_ratio', 'consent_deferred_ratio',
        'resource_conflict_counter_goal_accomplishment_ratio', 'avg_steps_overall'
    ]
    
    # Calculate mean and standard error for each metric
    grouped = df.groupby(['experiment_name', 'agent_config'])
    
    mean_df = grouped[metrics_to_average].mean().reset_index()

    return mean_df, df

mean_df, df = create_agent_ratio_analysis(experiment_name="goal_vs_consent_based_analysis", experiment_date="20251107")


Analyzing experiment: goal_vs_consent_based_analysis, date: 20251107
Figures will be saved to: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/figures
Found experiment-specific configs in: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/configs
Found matching configs in subdirectory: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/configs (110 files)


##  Graph 01-02: 1-way ANOVA: Accomplished Goals

In [5]:
df_sorted = df.drop_duplicates().sort_values(by=['goal_first_count', 'seed'], ascending=True)
df_sorted["consent_violation_ratio"] = df_sorted["violated_consents"] / df_sorted["total_consents"]
groups = [
    df_sorted[df_sorted["goal_first_count"] == r]["accomplished_goals"]
    for r in sorted(df_sorted["goal_first_count"].unique())
]

F , p = stats.f_oneway(*groups)
F, p

# Get min max values of the averages graph:
max_accomplished_goals = mean_df.groupby("goal_first_count")["accomplished_goals"].mean().max()
min_accomplished_goals = mean_df.groupby("goal_first_count")["accomplished_goals"].mean().min()

print(f"F: {F}, p: {p}")
print(f"Max: {max_accomplished_goals}, Min: {min_accomplished_goals}")




F: 112.93136737356, p: 1.5979920851030701e-49
Max: 2997.7, Min: 512.1


## Graph 03: 1-way ANOVA: Consent Violation Ratio

In [6]:
groups = [
    df_sorted[df_sorted["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_sorted["goal_first_count"].unique())
]

F , p = stats.f_oneway(*groups)
F, p

max_consent_violation_ratio = mean_df.groupby("goal_first_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio = mean_df.groupby("goal_first_count")["consent_violation_ratio"].mean().min()

print(f"F: {F}, p: {p}")
print(f"Max: {max_consent_violation_ratio}, Min: {min_consent_violation_ratio}")

F: 45.612388368024945, p: 1.2198597720224293e-32
Max: 0.7435923562503144, Min: 0.4099140920086988


## Get Agent Level Data

In [7]:

def create_agent_level_analysis(experiment_name=None, experiment_date=None):
    """Create analysis of agent-level metrics comparing ConsentFirstAgent and GoalFirstAgent.
    
    This function shows how individual agent performance varies across different configurations.
    Uses agent CSV files and config JSON files directly, no model CSV files.
    """
    print(f"\nCreating Agent-Level Analysis for: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    print(f"Figures will be saved to: {figures_dir}")
    
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    def _find_agent_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_agents.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_agents.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    def _find_model_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_model.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_model.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    # Collect agent-level data from all agent CSV files
    agent_data_list = []
    final_agent_values_list = []
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config to get agent counts
        try:
            with open(config_file, 'r') as f:
                config = json.load(f)
        except Exception as e:
            print(f"Warning: Could not load config file {config_file}: {e}")
            continue
        
        # Extract agent counts from config
        params = config.get('parameters', {})
        consent_first = params.get('ConsentFirstAgent_COUNT', 0)
        goal_first = params.get('GoalFirstAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = consent_first + goal_first + fifty_fifty
        
        # Get config name (without _config.json suffix)
        config_name = config_file.stem
        
        # Find corresponding agent file
        prefix = config_name.rsplit('_', 1)[0]
        agent_file = _find_agent_file(prefix)
        model_file = _find_model_file(prefix)
        
        if agent_file is None or not agent_file.exists():
            print(f"Warning: Agent file not found for {prefix}")
            continue
        
        if model_file is None or not model_file.exists():
            print(f"Warning: Model file not found for {prefix}")
            continue
        
        try:
            agent_df = pd.read_csv(agent_file)
            model_df = pd.read_csv(model_file)
            
            # Get the step column
            step_col = 'Step' if 'Step' in agent_df.columns else agent_df.columns[0]

            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]

                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
            
            # Get final step and calculate avg_steps_overall from agent CSV
            steps = pd.to_numeric(agent_df[step_col], errors='coerce')
            last_step = steps.max()
            avg_steps_overall = int(last_step) if not pd.isna(last_step) else np.nan
            
            final_agent_values = agent_df[steps == last_step].copy()
            
            if 'Agent Persona' not in final_agent_values.columns:
                print(f"Warning: 'Agent Persona' column not found in {agent_file}")
                continue
            
            # Create masks for agent types
            consent_first_mask = final_agent_values['Agent Persona'] == 'ConsentFirstAgent'
            goal_first_mask = final_agent_values['Agent Persona'] == 'GoalFirstAgent'

            final_agent_values["seed"] = seed
            final_agent_values["agent_config"] = agent_config
            
            agent_data_list.append(final_agent_values)
        except Exception as e:
            print(f"Error processing {prefix}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if len(agent_data_list) == 0:
        print("Warning: No agent data collected. Returning None.")
        return None
    
    all_agent_values_df = pd.concat(agent_data_list)
    all_agent_values_df["goal_first_count"] = all_agent_values_df["agent_config"].str.split("-").str[1].astype(int)
    return all_agent_values_df


In [8]:
final_agent_values = create_agent_level_analysis(experiment_name="goal_vs_consent_based_analysis", experiment_date="20251107")
#agent_mean_df
final_agent_values


Creating Agent-Level Analysis for: goal_vs_consent_based_analysis, date: 20251107
Figures will be saved to: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/figures
Found experiment-specific configs in: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/configs
Found matching configs in subdirectory: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/configs (110 files)


,Step,AgentID,Agent Persona,Accomplished Goals,Remaining Goals,Resource Conflicts,Counter Conflict Goal Accomplishments,Finished Step,Longest Idle Time,Total Idle Time,...,Number of Consents as G,Number of Consents as R Violated,Number of Consents as R Fulfilled,Number of Consents as R Unrealized,Number of Consents as G Violated,Number of Consents as G Fulfilled,Number of Consents as G Unrealized,seed,agent_config,goal_first_count
48000,48,1,ConsentFirstAgent,3,0,27,0,34.0,26,31,...,14,7,8,0,7,7,0,999,800-200-0-0,200
48001,48,2,ConsentFirstAgent,3,0,0,0,8.0,5,5,...,15,1,11,0,7,8,0,999,800-200-0-0,200
48002,48,3,ConsentFirstAgent,3,0,22,0,33.0,28,30,...,14,6,10,0,11,3,0,999,800-200-0-0,200
48003,48,4,ConsentFirstAgent,3,0,28,1,36.0,23,33,...,11,18,8,0,5,6,0,999,800-200-0-0,200
48004,48,5,ConsentFirstAgent,3,0,25,0,34.0,31,31,...,18,4,6,0,8,10,0,999,800-200-0-0,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,11,996,GoalFirstAgent,1,2,0,0,NaN,10,10,...,1,0,3,0,1,0,0,42,0-1000-0-0,1000
11996,11,997,GoalFirstAgent,1,2,7,0,NaN,10,10,...,2,3,2,0,2,0,0,42,0-1000-0-0,1000
11997,11,998,GoalFirstAgent,0,3,7,0,NaN,11,11,...,4,3,0,0,2,2,0,42,0-1000-0-0,1000
11998,11,999,GoalFirstAgent,0,3,7,0,NaN,11,11,...,4,6,0,0,2,2,0,42,0-1000-0-0,1000


In [9]:
final_agent_values

,Step,AgentID,Agent Persona,Accomplished Goals,Remaining Goals,Resource Conflicts,Counter Conflict Goal Accomplishments,Finished Step,Longest Idle Time,Total Idle Time,...,Number of Consents as G,Number of Consents as R Violated,Number of Consents as R Fulfilled,Number of Consents as R Unrealized,Number of Consents as G Violated,Number of Consents as G Fulfilled,Number of Consents as G Unrealized,seed,agent_config,goal_first_count
48000,48,1,ConsentFirstAgent,3,0,27,0,34.0,26,31,...,14,7,8,0,7,7,0,999,800-200-0-0,200
48001,48,2,ConsentFirstAgent,3,0,0,0,8.0,5,5,...,15,1,11,0,7,8,0,999,800-200-0-0,200
48002,48,3,ConsentFirstAgent,3,0,22,0,33.0,28,30,...,14,6,10,0,11,3,0,999,800-200-0-0,200
48003,48,4,ConsentFirstAgent,3,0,28,1,36.0,23,33,...,11,18,8,0,5,6,0,999,800-200-0-0,200
48004,48,5,ConsentFirstAgent,3,0,25,0,34.0,31,31,...,18,4,6,0,8,10,0,999,800-200-0-0,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,11,996,GoalFirstAgent,1,2,0,0,NaN,10,10,...,1,0,3,0,1,0,0,42,0-1000-0-0,1000
11996,11,997,GoalFirstAgent,1,2,7,0,NaN,10,10,...,2,3,2,0,2,0,0,42,0-1000-0-0,1000
11997,11,998,GoalFirstAgent,0,3,7,0,NaN,11,11,...,4,3,0,0,2,2,0,42,0-1000-0-0,1000
11998,11,999,GoalFirstAgent,0,3,7,0,NaN,11,11,...,4,6,0,0,2,2,0,42,0-1000-0-0,1000


In [ ]:
df_sorted = final_agent_values.drop_duplicates().sort_values(by=['goal_first_count', 'seed', 'Step'], ascending=True)
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]
df_sorted["consent_violation_ratio"] = df_sorted["Number of Consents as R Violated"] / df_sorted["Number of Consents as R"]
df_sorted["consent_fulfillment_ratio"] = df_sorted["Number of Consents as R Fulfilled"] / df_sorted["Number of Consents as R"]

df_da = df_sorted[df_sorted["Agent Persona"] == "ConsentFirstAgent"]
df_ta = df_sorted[df_sorted["Agent Persona"] == "GoalFirstAgent"]



,Step,AgentID,Agent Persona,Accomplished Goals,Remaining Goals,Resource Conflicts,Counter Conflict Goal Accomplishments,Finished Step,Longest Idle Time,Total Idle Time,...,Number of Consents as R Fulfilled,Number of Consents as R Unrealized,Number of Consents as G Violated,Number of Consents as G Fulfilled,Number of Consents as G Unrealized,seed,agent_config,goal_first_count,consent_violation_ratio,consent_fulfillment_ratio
34000,34,1,ConsentFirstAgent,3,0,17,2,26.0,23,23,...,10,0,8,6,0,123,1000-0-0-0,0,0.545455,0.454545
34001,34,2,ConsentFirstAgent,3,0,0,0,6.0,3,3,...,7,0,5,8,0,123,1000-0-0-0,0,0.000000,1.000000
34002,34,3,ConsentFirstAgent,3,0,7,0,15.0,12,12,...,9,0,3,6,0,123,1000-0-0-0,0,0.250000,0.750000
34003,34,4,ConsentFirstAgent,3,0,11,3,27.0,13,24,...,9,0,10,14,0,123,1000-0-0-0,0,0.526316,0.473684
34004,34,5,ConsentFirstAgent,3,0,8,2,15.0,12,12,...,9,0,7,10,0,123,1000-0-0-0,0,0.500000,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16094,16,95,ConsentFirstAgent,0,3,11,0,NaN,16,16,...,0,0,3,2,0,999,100-900-0-0,900,0.750000,0.000000
16095,16,96,ConsentFirstAgent,0,3,0,0,NaN,16,16,...,0,0,2,5,0,999,100-900-0-0,900,0.727273,0.000000
16096,16,97,ConsentFirstAgent,0,3,11,0,NaN,16,16,...,0,0,3,1,0,999,100-900-0-0,900,0.714286,0.000000
16098,16,99,ConsentFirstAgent,1,2,0,0,NaN,14,15,...,1,0,4,2,0,999,100-900-0-0,900,0.666667,0.083333


## Consent Violation Ratio, 1-WAY ANOVA TESTS

In [19]:
groups_da = [
    df_da[df_da["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_da["goal_first_count"].unique())
]

F , p = stats.f_oneway(*groups_da)
print(f"F: {F}, p: {p}")

groups_ta = [
    df_ta[df_ta["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_ta["goal_first_count"].unique())
]

F , p = stats.f_oneway(*groups_ta)
print(f"F: {F}, p: {p}")

F: 3110.4077113851495, p: 0.0
F: 625.6822448845777, p: 0.0


## Consent Fulfilment Ratio, 1-WAY ANOVA TESTS

In [22]:
groups_da = [
    df_da[df_da["goal_first_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_da["goal_first_count"].unique())
]

F , p = stats.f_oneway(*groups_da)
print(f"F: {F}, p: {p}")

groups_ta = [
    df_ta[df_ta["goal_first_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_ta["goal_first_count"].unique())
]

F , p = stats.f_oneway(*groups_ta)
print(f"F: {F}, p: {p}")

F: 4671.042172883021, p: 0.0
F: 705.4113011493633, p: 0.0


In [11]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Make sure categorical fields are treated as categorical
df_sorted["agent_persona_cat"] = df_sorted["Agent Persona"].astype("category")
df_sorted["ratio_cat"] = df_sorted["goal_first_count"].astype("category")

# Fit model
model = ols(
    "consent_violation_ratio ~ C(agent_persona_cat) * C(ratio_cat)",
    data=df_sorted
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                                         sum_sq        df             F  \
C(agent_persona_cat)              -1.411648e-08       1.0 -2.095623e-07   
C(ratio_cat)                       8.727710e+02      10.0  1.295648e+03   
C(agent_persona_cat):C(ratio_cat)  1.031930e+00      10.0  1.531923e+00   
Residual                           7.277627e+03  108038.0           NaN   

                                     PR(>F)  
C(agent_persona_cat)               1.000000  
C(ratio_cat)                       0.000000  
C(agent_persona_cat):C(ratio_cat)  0.216124  
Residual                                NaN  


/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 10, but rank is 2
  warnings.warn('covariance of constraints does not have full '


## 04: Consent Violation Ratio Mann-Whitney Test


In [12]:
# Compare violation ratios between ConsentFirstAgent and GoalFirstAgent
# Check if one persona consistently has higher violation ratio than the other

# Use the same filtered data as before
df_sorted = final_agent_values.drop_duplicates().sort_values(by=['goal_first_count', 'seed', 'Step'], ascending=True)
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]
df_sorted["consent_violation_ratio"] = df_sorted["Number of Consents as R Violated"] / df_sorted["Number of Consents as R"]

# Calculate mean violation ratio for each persona across all conditions
consent_first_means = df_sorted[df_sorted["Agent Persona"] == "ConsentFirstAgent"].groupby("goal_first_count")["consent_violation_ratio"].mean()
goal_first_means = df_sorted[df_sorted["Agent Persona"] == "GoalFirstAgent"].groupby("goal_first_count")["consent_violation_ratio"].mean()

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'goal_first_count': consent_first_means.index,
    'ConsentFirstAgent_mean': consent_first_means.values,
    'GoalFirstAgent_mean': goal_first_means.values
})
comparison_df['Difference'] = comparison_df['ConsentFirstAgent_mean'] - comparison_df['GoalFirstAgent_mean']
comparison_df['ConsentFirst_higher'] = comparison_df['Difference'] > 0

print("Violation Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("\n" + "=" * 80)

# Count how many conditions each persona has higher violation ratio
consent_first_higher_count = comparison_df['ConsentFirst_higher'].sum()
goal_first_higher_count = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent has higher violation ratio in {consent_first_higher_count} out of {len(comparison_df)} conditions")
print(f"GoalFirstAgent has higher violation ratio in {goal_first_higher_count} out of {len(comparison_df)} conditions")

# Overall mean comparison
overall_consent_first = df_sorted[df_sorted["Agent Persona"] == "ConsentFirstAgent"]["consent_violation_ratio"].mean()
overall_goal_first = df_sorted[df_sorted["Agent Persona"] == "GoalFirstAgent"]["consent_violation_ratio"].mean()

print(f"\nOverall mean violation ratio:")
print(f"  ConsentFirstAgent: {overall_consent_first:.4f}")
print(f"  GoalFirstAgent: {overall_goal_first:.4f}")
print(f"  Difference: {overall_consent_first - overall_goal_first:.4f}")

# Statistical test: Mann-Whitney U test (non-parametric, doesn't assume normal distribution)
from scipy.stats import mannwhitneyu

consent_first_ratios = df_sorted[df_sorted["Agent Persona"] == "ConsentFirstAgent"]["consent_violation_ratio"]
goal_first_ratios = df_sorted[df_sorted["Agent Persona"] == "GoalFirstAgent"]["consent_violation_ratio"]

u_statistic, p_value = mannwhitneyu(consent_first_ratios, goal_first_ratios, alternative='two-sided')
print(f"\nMann-Whitney U Test (two-sided):")
print(f"  U-statistic: {u_statistic:.2f}")
print(f"  p-value: {p_value:.2e}")

if p_value < 0.05:
    if overall_consent_first > overall_goal_first:
        print(f"  Result: ConsentFirstAgent has significantly higher violation ratio (p < 0.05)")
    else:
        print(f"  Result: GoalFirstAgent has significantly higher violation ratio (p < 0.05)")
else:
    print(f"  Result: No significant difference between personas (p >= 0.05)")

# One-sided test to check if ConsentFirstAgent is consistently higher
u_statistic_one_sided, p_value_one_sided = mannwhitneyu(
    consent_first_ratios, goal_first_ratios, alternative='greater'
)
print(f"\nMann-Whitney U Test (one-sided: ConsentFirstAgent > GoalFirstAgent):")
print(f"  U-statistic: {u_statistic_one_sided:.2f}")
print(f"  p-value: {p_value_one_sided:.2e}")
if p_value_one_sided < 0.05:
    print(f"  Result: ConsentFirstAgent has significantly higher violation ratio")
else:
    print(f"  Result: No evidence that ConsentFirstAgent has higher violation ratio")

# One-sided test to check if GoalFirstAgent is consistently higher
u_statistic_one_sided2, p_value_one_sided2 = mannwhitneyu(
    goal_first_ratios, consent_first_ratios, alternative='greater'
)
print(f"\nMann-Whitney U Test (one-sided: GoalFirstAgent > ConsentFirstAgent):")
print(f"  U-statistic: {u_statistic_one_sided2:.2f}")
print(f"  p-value: {p_value_one_sided2:.2e}")
if p_value_one_sided2 < 0.05:
    print(f"  Result: GoalFirstAgent has significantly higher violation ratio")
else:
    print(f"  Result: No evidence that GoalFirstAgent has higher violation ratio")


Violation Ratio Comparison by Persona:
 goal_first_count  ConsentFirstAgent_mean  GoalFirstAgent_mean  Difference  ConsentFirst_higher
                0                0.369223             0.409748   -0.040525                False
              100                0.406739             0.425505   -0.018765                False
              200                0.453605             0.510084   -0.056478                False
              300                0.546051             0.690975   -0.144924                False
              400                0.701948             0.775057   -0.073109                False
              500                0.754614             0.779197   -0.024583                False
              600                0.756323             0.783840   -0.027517                False
              700                0.730357             0.774804   -0.044447                False
              800                0.693197             0.770276   -0.077079                False
 

## 05: Consent Fulfilment Ratio Mann-Whitney Test


In [23]:
# Compare violation ratios between ConsentFirstAgent and GoalFirstAgent
# Check if one persona consistently has higher violation ratio than the other

# Use the same filtered data as before
df_sorted = final_agent_values.drop_duplicates().sort_values(by=['goal_first_count', 'seed', 'Step'], ascending=True)
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]
df_sorted["consent_fulfilment_ratio"] = df_sorted["Number of Consents as R Fulfilled"] / df_sorted["Number of Consents as R"]

# Calculate mean violation ratio for each persona across all conditions
consent_first_means = df_sorted[df_sorted["Agent Persona"] == "ConsentFirstAgent"].groupby("goal_first_count")["consent_fulfilment_ratio"].mean()
goal_first_means = df_sorted[df_sorted["Agent Persona"] == "GoalFirstAgent"].groupby("goal_first_count")["consent_fulfilment_ratio"].mean()

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'goal_first_count': consent_first_means.index,
    'ConsentFirstAgent_mean': consent_first_means.values,
    'GoalFirstAgent_mean': goal_first_means.values
})
comparison_df['Difference'] = comparison_df['ConsentFirstAgent_mean'] - comparison_df['GoalFirstAgent_mean']
comparison_df['ConsentFirst_higher'] = comparison_df['Difference'] > 0

print("Fulfilment Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("\n" + "=" * 80)

# Count how many conditions each persona has higher violation ratio
consent_first_higher_count = comparison_df['ConsentFirst_higher'].sum()
goal_first_higher_count = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent has higher fulfilment ratio in {consent_first_higher_count} out of {len(comparison_df)} conditions")
print(f"GoalFirstAgent has higher fulfilment ratio in {goal_first_higher_count} out of {len(comparison_df)} conditions")

# Overall mean comparison
overall_consent_first = df_sorted[df_sorted["Agent Persona"] == "ConsentFirstAgent"]["consent_fulfilment_ratio"].mean()
overall_goal_first = df_sorted[df_sorted["Agent Persona"] == "GoalFirstAgent"]["consent_fulfilment_ratio"].mean()

print(f"\nOverall mean fulfilment ratio:")
print(f"  ConsentFirstAgent: {overall_consent_first:.4f}")
print(f"  GoalFirstAgent: {overall_goal_first:.4f}")
print(f"  Difference: {overall_consent_first - overall_goal_first:.4f}")

# Statistical test: Mann-Whitney U test (non-parametric, doesn't assume normal distribution)
from scipy.stats import mannwhitneyu

consent_first_ratios = df_sorted[df_sorted["Agent Persona"] == "ConsentFirstAgent"]["consent_fulfilment_ratio"]
goal_first_ratios = df_sorted[df_sorted["Agent Persona"] == "GoalFirstAgent"]["consent_fulfilment_ratio"]

u_statistic, p_value = mannwhitneyu(consent_first_ratios, goal_first_ratios, alternative='two-sided')
print(f"\nMann-Whitney U Test (two-sided):")
print(f"  U-statistic: {u_statistic:.2f}")
print(f"  p-value: {p_value:.2e}")

if p_value < 0.05:
    if overall_consent_first > overall_goal_first:
        print(f"  Result: ConsentFirstAgent has significantly higher fulfilment ratio (p < 0.05)")
    else:
        print(f"  Result: GoalFirstAgent has significantly higher fulfilment ratio (p < 0.05)")
else:
    print(f"  Result: No significant difference between personas (p >= 0.05)")

# One-sided test to check if ConsentFirstAgent is consistently higher
u_statistic_one_sided, p_value_one_sided = mannwhitneyu(
    consent_first_ratios, goal_first_ratios, alternative='greater'
)
print(f"\nMann-Whitney U Test (one-sided: ConsentFirstAgent > GoalFirstAgent):")
print(f"  U-statistic: {u_statistic_one_sided:.2f}")
print(f"  p-value: {p_value_one_sided:.2e}")
if p_value_one_sided < 0.05:
    print(f"  Result: ConsentFirstAgent has significantly higher fulfilment ratio")
else:
    print(f"  Result: No evidence that ConsentFirstAgent has higher fulfilment ratio")

# One-sided test to check if GoalFirstAgent is consistently higher
u_statistic_one_sided2, p_value_one_sided2 = mannwhitneyu(
    goal_first_ratios, consent_first_ratios, alternative='greater'
)
print(f"\nMann-Whitney U Test (one-sided: GoalFirstAgent > ConsentFirstAgent):")
print(f"  U-statistic: {u_statistic_one_sided2:.2f}")
print(f"  p-value: {p_value_one_sided2:.2e}")
if p_value_one_sided2 < 0.05:
    print(f"  Result: GoalFirstAgent has significantly higher fulfilment ratio")
else:
    print(f"  Result: No evidence that GoalFirstAgent has higher fulfilment ratio")



Fulfilment Ratio Comparison by Persona:
 goal_first_count  ConsentFirstAgent_mean  GoalFirstAgent_mean  Difference  ConsentFirst_higher
                0                0.630631             0.590252    0.040379                 True
              100                0.591393             0.574353    0.017040                 True
              200                0.543518             0.489582    0.053935                 True
              300                0.443421             0.307987    0.135434                 True
              400                0.241310             0.223505    0.017805                 True
              500                0.162488             0.218459   -0.055971                False
              600                0.146801             0.211259   -0.064458                False
              700                0.143420             0.212789   -0.069369                False
              800                0.149941             0.215865   -0.065924                False


## 04: Consent Violation Ratio Mann-Whitney Test
